In [ ]:
from huggingface_hub import HfApi
import os
import random
import shutil
from pathlib import Path
import soundfile as sf
import numpy as np
import yaml
from datasets import load_dataset

In [ ]:
api = HfApi()

In [ ]:
# Paths
source_dir = Path("/home/an/Documents/Latent-DSB-Data/rir")
target_dir = Path("/home/an/Documents/Latent-DSB-Data/rir_prepared")

# Configuration
test_split_ratio = 0.10
sub_datasets = [f.name for f in source_dir.iterdir() if f.is_dir()]

# Reset target directory
if target_dir.exists():
    shutil.rmtree(target_dir)
target_dir.mkdir(parents=True, exist_ok=True)

def split_and_copy(files, target_path, prefix="rir"):
    """Shuffles, splits, and copies files with zfill naming."""
    if not files:
        return
    
    random.shuffle(files)
    split_idx = int(len(files) * (1 - test_split_ratio))
    train_files = files[:split_idx]
    test_files = files[split_idx:]

    # Create directories
    (target_path / "train").mkdir(parents=True, exist_ok=True)
    (target_path / "test").mkdir(parents=True, exist_ok=True)

    # Copy Train
    for i, f in enumerate(train_files):
        new_name = f"{prefix}_{str(i).zfill(4)}.wav"
        shutil.copy2(f, target_path / "train" / new_name)

    # Copy Test (Continuous indexing)
    start_test = len(train_files)
    for i, f in enumerate(test_files):
        new_name = f"{prefix}_{str(i + start_test).zfill(4)}.wav"
        shutil.copy2(f, target_path / "test" / new_name)

for dataset in sub_datasets:
    dataset_source = source_dir / dataset
    print(f"--- Processing {dataset} ---")
    
    all_wavs = list(dataset_source.rglob("*.wav"))
    mono_files = []
    #in general, more than 1 channel
    binaural_files = []
    
    # 1. Sort files by channel count
    for f in all_wavs:
        try:
            info = sf.info(f)
            if info.channels > 1:
                binaural_files.append(f)
                #print(f,info.channels)
            else:
                mono_files.append(f)
                if info.channels != 1:
                    print(f,info.channels)
        except Exception as e:
            print(f"Error reading {f.name}: {e}")

    # 2. Process Mono Files (rir_prepared/DatasetName/train|test)
    if mono_files:
        split_and_copy(mono_files, target_dir / dataset)

    # 3. Process Binaural Files (rir_prepared/binaural_rirs/DatasetName/train|test)
    if binaural_files:
        split_and_copy(binaural_files, target_dir / "binaural_rirs" / dataset)

    print(f"Done {dataset}: {len(mono_files)} Mono, {len(binaural_files)} Binaural.")

print(f"\nSuccess! Prepared dataset is at: {target_dir}")

In [ ]:
# --- Configuration ---
prepared_dir = Path("/home/an/Documents/Latent-DSB-Data/rir_prepared")
repo_id = "andnymand/RIR-datasets"
validation_corpus_name = "MIT_Survey"  # <--- Change this to your validation folder name

# --- 1. Logic for Mono Config ---
m_train_paths = []
m_test_paths = []
m_val_paths = []

for folder in prepared_dir.iterdir():
    # Ignore binaural and non-directories
    if not folder.is_dir() or folder.name == "binaural_rirs":
        continue
        
    rel = folder.relative_to(prepared_dir)
    
    if folder.name == validation_corpus_name:
        # Move BOTH splits of this specific corpus to Validation
        if (folder / "train").exists(): m_val_paths.append(f"{rel}/train/*")
        if (folder / "test").exists(): m_val_paths.append(f"{rel}/test/*")
    else:
        # Standard Train/Test for the other 4 corpora
        if (folder / "train").exists(): m_train_paths.append(f"{rel}/train/*")
        if (folder / "test").exists(): m_test_paths.append(f"{rel}/test/*")

# --- 2. Logic for Binaural Config (Keeping standard splits) ---
b_train_paths = []
b_test_paths = []
binaural_root = prepared_dir / "binaural_rirs"

if binaural_root.exists():
    for folder in binaural_root.iterdir():
        if folder.is_dir():
            rel = folder.relative_to(prepared_dir)
            if (folder / "train").exists(): b_train_paths.append(f"{rel}/train/*")
            if (folder / "test").exists(): b_test_paths.append(f"{rel}/test/*")

# --- 3. Construct YAML ---
yaml_data = {
    "configs": [
        {
            "config_name": "mono",
            "data_files": [
                {"split": "train", "path": m_train_paths},
                {"split": "test", "path": m_test_paths},
                {"split": "validation", "path": m_val_paths}
            ]
        },
        {
            "config_name": "binaural",
            "data_files": [
                {"split": "train", "path": b_train_paths},
                {"split": "test", "path": b_test_paths}
            ]
        }
    ]
}

# Write README.md
readme_text = f"---\n{yaml.dump(yaml_data, sort_keys=False)}---\n"
readme_text += "# RIR Datasets\n\nThis repository contains standardized Room Impulse Responses (RIRs) from 5 different open-source RIR corpora"
readme_text +="\nThe RIRs have been recursively filtered by channel dimension to filter binaural and monaural RIRs."
readme_text += f"\n One of the corpora has been uploaded as a validation set ({validation_corpus_name})"

# Save README.md to the prepared folder
with open(prepared_dir / "README.md", "w") as f:
    f.write(readme_text)

# --- 4. Upload ---
print(f"Uploading with {validation_corpus_name} as the validation split...")
HfApi().upload_large_folder(folder_path=str(prepared_dir), repo_id=repo_id, repo_type="dataset")

In [ ]:
dataset = load_dataset(
    "andnymand/RIR-datasets",
    "binaural",
    split="train",
    cache_dir="/home/an/Documents/Latent-DSB-Data/hf_cache",
)

In [ ]:
print(f"Raw array shape: {dataset[2]['audio']['array'].shape}")